# Dataset of the molecules clustered using the dictionnary 
## Two variants: **A-keep-odereless** vs **A-drop-odereless**

Base transformation 
1. Drop the 8 same-named fine labels: `citrus, earthy, floral, fruity, green, spicy, sweet, woody`
2. Keep their meta columns unchanged (This way we have a  partial independence of the parent from the children)


Odorless strategy differs between variants:

| | molecules | fine labels | meta labels |
|---|---|---|---|
| **A-keep** | 4976 | 129 | 13 (odorless added as childless meta) |
| **A-drop** | 4777 | 129 | 12 (odorless molecules removed) |


In [9]:
import pandas as pd

df = pd.read_csv(r"C:\Users\hp\Desktop\stage etis\HMCN\hmcn_dataset.csv")

fine_cols = [c for c in df.columns if c.startswith("fine_")]
meta_cols = [c for c in df.columns if c.startswith("meta_")]
feat_cols = [c for c in df.columns if c not in fine_cols + meta_cols + ["SMILES"]]

print(f"Loaded: {len(df)} molecules | {len(feat_cols)} features | {len(fine_cols)} fine | {len(meta_cols)} meta")

Loaded: 4976 molecules | 1005 features | 138 fine | 12 meta


## Drop the 8 same-named fine labels
The 216 molecules that had only one of these labels
become meta-only 

In [10]:
meta_names = {c.replace("meta_", "") for c in meta_cols}
fine_names = {c.replace("fine_", "") for c in fine_cols}
overlap    = sorted(meta_names & fine_names)
drop_same  = ["fine_" + n for n in overlap]

df = df.drop(columns=drop_same)

fine_now = [c for c in df.columns if c.startswith("fine_")]
meta_only_count = (df[fine_now].sum(axis=1) == 0).sum()
print(f"Dropped {len(drop_same)} same-named fine labels: {overlap}")
print(f"Remaining fine labels : {len(fine_now)}")
print(f"Meta-only molecules   : {meta_only_count} ")

Dropped 8 same-named fine labels: ['citrus', 'earthy', 'floral', 'fruity', 'green', 'spicy', 'sweet', 'woody']
Remaining fine labels : 130
Meta-only molecules   : 216 


## Identify odorless molecules

In [11]:
odorless_mask = df["fine_odorless"] == 1
other_sum     = df[[c for c in fine_now if c != "fine_odorless"]].sum(axis=1)
purely_odorless = ((odorless_mask) & (other_sum == 0)).sum()
mixed           = ((odorless_mask) & (other_sum > 0)).sum()

print(f"Odorless total         : {odorless_mask.sum()}")
print(f"  purely odorless      : {purely_odorless} ")
print(f"  odorless + other fine: {mixed}             ")

Odorless total         : 200
  purely odorless      : 199 
  odorless + other fine: 1             


---
## Step 3 — Odorless strategy

### ▶  KEEP ODERLESS


fine_odorless is dropped and replaced by meta_odorless as a childless metacategory.  


In [ ]:
# KEEP

''' dataset = df.drop(columns=["fine_odorless"]).copy()
dataset["meta_odorless"] = odorless_mask.astype(int)

fine_final = [c for c in dataset.columns if c.startswith("fine_")]
meta_final = [c for c in dataset.columns if c.startswith("meta_")]
print(f"A-KEEP -> {len(dataset)} molecules | {len(fine_final)} fine | {len(meta_final)} meta")

dataset.to_csv("hmcn_dataset_A_keep.csv", index=False)
print("Saved -> hmcn_dataset_A_keep.csv" )'''
# end KEEP

A-KEEP -> 4976 molecules | 129 fine | 13 meta
Saved -> hmcn_dataset_A_keep.csv


###  DROP ODERLESS

`fine_odorless` and the 199 purely-odorless molecules are removed.  

In [12]:
# DROP ODERLESS: 

drop_rows = df.index[(odorless_mask) & (other_sum == 0)]
dataset   = df.drop(index=drop_rows).drop(columns=["fine_odorless"]).reset_index(drop=True)

fine_final = [c for c in dataset.columns if c.startswith("fine_")]
meta_final = [c for c in dataset.columns if c.startswith("meta_")]
print(f"Dropped {len(drop_rows)} purely-odorless molecules")
print(f"A-DROP -> {len(dataset)} molecules | {len(fine_final)} fine | {len(meta_final)} meta")

dataset.to_csv("hmcn_dataset_A_drop.csv", index=False)
print("Saved -> hmcn_dataset_A_drop.csv")
# end DROP ODERLESS

Dropped 199 purely-odorless molecules
A-DROP -> 4777 molecules | 129 fine | 12 meta
Saved -> hmcn_dataset_A_drop.csv
